### Notebook : 06_vector_search_agent


##### 1. Notebook Purpose

The Vector Search Agent retrieves semantically similar customer notes when requested by the Coordinator Agent.

It:

- Reads the Coordinator execution plan.
- Finds the task assigned to vector_search_agent.
- Validates task dependencies.
- Uses the task description or user request as the semantic-search query.
- Calls the injected Vector Search Tool.
- Validates and normalizes the tool response.
- Creates a validated VectorSearchAgentResult.
- Stores the result in shared state.
- Records execution history and errors.
- Skips cleanly when no Vector Search task is assigned.

The Vector Search Agent does not decide whether semantic search is needed. That decision belongs to the Coordinator Agent.


##### 2. Technologies Used

- Python
- Databricks Vector Search
- Embedding models
- Semantic search
- Pydantic
- TypedDict
- Dependency injection
- Shared multi-agent state
- Unit testing with mock functions


##### 3. Input

- state: MultiAgentState
- vector_search_tool: VectorSearchToolFunction
- The shared state should contain:

    - state["user_request"]
    - state["coordinator_result"]
    - state["agent_results"]
    - state["execution_history"]
    - state["errors"]


###### 4. Output

Updated

MultiAgentState containing state["agent_results"]["vector_search_agent"]


##### 5. Architecture

```text

Coordinator Agent
        │
        ▼
CoordinatorResult
        │
        ▼
Vector Search Agent
        │
        ├── Find assigned task
        ├── Validate task dependencies
        ├── Resolve semantic-search query
        ├── Call Vector Search Tool
        ├── Validate tool response
        ├── Create VectorSearchAgentResult
        └── Store result in shared state
        │
        ▼
Updated MultiAgentState

```


###### 6. Load Shared Models and Helpers

In [0]:
%run ./01_shared_models_code_only

In [0]:
%run ./02_shared_state_and_helpers_code_only


##### 7. Imports

In [0]:
from typing import Any, Callable, Dict, List, Optional


##### 8. Vector Search Tool Contract

In [0]:
VectorSearchToolFunction = Callable[
    [str],
    Dict[str, Any]
]


##### 9. Agent Constants

In [0]:
VECTOR_SEARCH_AGENT_NAME = "vector_search_agent"

SUCCESS_STATUS = "success"
ERROR_STATUS = "error"
SKIPPED_STATUS = "skipped"

DEFAULT_NUM_RESULTS = 3

##### 10. Find the Vector Search Agent Task

In [0]:
# This function searches the Coordinator execution plan for a task assigned to the Vector Search Agent.

def find_vector_search_agent_task(
    coordinator_result: CoordinatorResult,
) -> Optional[AgentTask]:
    """
    Find the task assigned to the Vector Search Agent.

    Parameters
    ----------
    coordinator_result:
        Validated Coordinator Agent result.

    Returns
    -------
    Optional[AgentTask]
        The assigned task, or None when the Vector Search Agent
        is not required.
    """

    for task in coordinator_result.execution_plan:
        if task.agent_name == VECTOR_SEARCH_AGENT_NAME:
            return task

    return None


##### 11. Resolve the Semantic-Search Query

In [0]:
def resolve_search_query(
    task: AgentTask,
    user_request: str,
) -> str:
    """
    Determine the query that should be sent to the
    Vector Search Tool.

    The task description is preferred. The original user request
    is used when the task description is empty.
    """

    task_description = task.task_description.strip()

    if task_description:
        return task_description

    normalized_user_request = user_request.strip()

    if normalized_user_request:
        return normalized_user_request

    raise ValueError(
        "Vector Search Agent could not determine a semantic-search query."
    )


##### 12. Convert Tool Values to Python Types

In [0]:
def convert_vector_search_value(
    value: Any,
) -> Any:
    """
    Convert Vector Search Tool values into standard Python objects.
    """

    if value is None:
        return None

    if hasattr(value, "asDict"):
        return {
            key: convert_vector_search_value(item)
            for key, item in value.asDict(
                recursive=True
            ).items()
        }

    if isinstance(value, dict):
        return {
            key: convert_vector_search_value(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            convert_vector_search_value(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return [
            convert_vector_search_value(item)
            for item in value
        ]

    if hasattr(value, "item"):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass

    return value


##### 13. Normalize Similarity Score

In [0]:
def normalize_similarity_score(
    score: Any,
) -> Optional[float]:
    """
    Normalize a similarity score to a float between 0.0 and 1.0.

    A missing score is allowed because some tool implementations
    may not expose it.
    """

    if score is None:
        return None

    score = convert_vector_search_value(score)

    try:
        normalized_score = float(score)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Invalid similarity score: {score!r}"
        ) from exc

    if not 0.0 <= normalized_score <= 1.0:
        raise ValueError(
            "Similarity score must be between 0.0 and 1.0."
        )

    return normalized_score


##### 14. Validate One Search Result

In [0]:
def validate_search_result(
    search_result: Dict[str, Any],
    result_position: int,
) -> Dict[str, Any]:
    """
    Validate and normalize one semantic-search result.
    """

    if not isinstance(search_result, dict):
        raise TypeError(
            f"Search result {result_position} must be a dictionary."
        )

    normalized_result = convert_vector_search_value(
        search_result
    )

    customer_id = normalized_result.get(
        "customer_id"
    )

    note = normalized_result.get(
        "note"
    )

    similarity_score = normalized_result.get(
        "similarity_score",
        normalized_result.get("score"),
    )

    if customer_id is None:
        raise ValueError(
            f"Search result {result_position} is missing "
            "'customer_id'."
        )

    if note is None:
        raise ValueError(
            f"Search result {result_position} is missing 'note'."
        )

    customer_id = str(customer_id).strip()
    note = str(note).strip()

    if not customer_id:
        raise ValueError(
            f"Search result {result_position} contains an empty "
            "customer ID."
        )

    if not note:
        raise ValueError(
            f"Search result {result_position} contains an empty note."
        )

    normalized_score = normalize_similarity_score(
        similarity_score
    )

    return {
        "customer_id": customer_id,
        "note": note,
        "similarity_score": normalized_score,
    }


##### 15. Validate Vector Search Tool Response

In [0]:
def validate_vector_search_tool_response(
    tool_response: Dict[str, Any],
    expected_query: str,
) -> Dict[str, Any]:
    """
    Validate and normalize the Vector Search Tool response.

    Parameters
    ----------
    tool_response:
        Raw dictionary returned by the Vector Search Tool.

    expected_query:
        Query originally submitted by the Vector Search Agent.

    Returns
    -------
    Dict[str, Any]
        Normalized query and search results.
    """

    if not isinstance(tool_response, dict):
        raise TypeError(
            "Vector Search Tool must return a dictionary."
        )

    normalized_response = convert_vector_search_value(
        tool_response
    )

    status = normalized_response.get("status")

    if status != SUCCESS_STATUS:
        error_message = normalized_response.get(
            "message",
            "Vector Search Tool execution failed.",
        )

        raise RuntimeError(error_message)

    returned_query = normalized_response.get(
        "query",
        expected_query,
    )

    returned_query = str(returned_query).strip()

    if not returned_query:
        returned_query = expected_query

    results = normalized_response.get("results")

    if results is None:
        raise ValueError(
            "Vector Search Tool response is missing 'results'."
        )

    if not isinstance(results, list):
        raise TypeError(
            "Vector Search Tool 'results' must be a list."
        )

    validated_results = [
        validate_search_result(
            search_result=result,
            result_position=index,
        )
        for index, result in enumerate(
            results,
            start=1,
        )
    ]

    return {
        "query": returned_query,
        "results": validated_results,
        "result_count": len(validated_results),
        "raw_tool_response": normalized_response,
    }


##### 16. Execute Semantic Search

In [0]:
def execute_semantic_search(
    query: str,
    vector_search_tool: VectorSearchToolFunction,
    num_results: int = DEFAULT_NUM_RESULTS,
) -> Dict[str, Any]:
    """
    Execute semantic search using the injected Vector Search Tool.
    """

    if not query.strip():
        raise ValueError(
            "Semantic-search query cannot be empty."
        )

    if num_results <= 0:
        raise ValueError(
            "num_results must be greater than zero."
        )

    return vector_search_tool(
        query,
        num_results,
    )


###### 17. Execute the Vector Search Agent

In [0]:
def execute_vector_search_agent(
    state: MultiAgentState,
    vector_search_tool: VectorSearchToolFunction,
    num_results: int = DEFAULT_NUM_RESULTS,
) -> Optional[VectorSearchAgentResult]:
    """
    Execute the task assigned to the Vector Search Agent.

    Returns None when no Vector Search task was assigned.
    """

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        raise ValueError(
            "Vector Search Agent cannot run because "
            "coordinator_result is missing."
        )

    task = find_vector_search_agent_task(
        coordinator_result=coordinator_result
    )

    if task is None:
        return None

    validate_task_dependencies(
        state=state,
        task=task,
    )

    user_request = state.get(
        "user_request",
        "",
    )

    query = resolve_search_query(
        task=task,
        user_request=user_request,
    )

    raw_tool_response = execute_semantic_search(
        query=query,
        vector_search_tool=vector_search_tool,
        num_results=num_results,
    )

    validated_response = (
        validate_vector_search_tool_response(
            tool_response=raw_tool_response,
            expected_query=query,
        )
    )

    results = validated_response["results"]
    result_count = validated_response[
        "result_count"
    ]

    if result_count == 0:
        message = (
            "Vector Search Agent completed successfully, "
            "but no similar customer notes were found."
        )
    else:
        message = (
            f"Vector Search Agent retrieved "
            f"{result_count} similar customer note"
            f"{'' if result_count == 1 else 's'}."
        )

    result = VectorSearchAgentResult(
        agent_name=VECTOR_SEARCH_AGENT_NAME,
        task_id=task.task_id,
        status=SUCCESS_STATUS,
        message=message,
        query=validated_response["query"],
        results=results,
    )

    return result

##### 18. Run the Vector Search Agent

In [0]:
def run_vector_search_agent(
    state: MultiAgentState,
    vector_search_tool: VectorSearchToolFunction,
    num_results: int = DEFAULT_NUM_RESULTS,
) -> MultiAgentState:
    """
    Run the Vector Search Agent and update shared state.
    """

    agent_name = VECTOR_SEARCH_AGENT_NAME

    try:
        result = execute_vector_search_agent(
            state=state,
            vector_search_tool=vector_search_tool,
            num_results=num_results,
        )

        if result is None:
            add_execution_history(
                state=state,
                agent_name=agent_name,
                status=SKIPPED_STATUS,
                message=(
                    "Vector Search Agent skipped because no "
                    "Vector Search task was assigned."
                ),
            )

            return state

        store_agent_result(
            state=state,
            agent_name=agent_name,
            result=result,
        )

        add_execution_history(
            state=state,
            agent_name=agent_name,
            status=SUCCESS_STATUS,
            message=result.message,
        )

    except Exception as exc:
        error_message = (
            f"Vector Search Agent failed: {exc}"
        )

        add_error(
            state=state,
            agent_name=agent_name,
            error_message=error_message,
        )

        add_execution_history(
            state=state,
            agent_name=agent_name,
            status=ERROR_STATUS,
            message=error_message,
        )

    return state

##### 19. Mock Vector Search Tools

In [0]:
# These mocks allow the Vector Search Agent to be unit tested without calling Databricks Vector Search or an embedding endpoint.

def mock_successful_vector_search_tool(
    query: str,
    num_results: int,
) -> Dict[str, Any]:
    """
    Return predictable semantic-search results.
    """

    all_results = [
        {
            "customer_id": "1001",
            "note": (
                "Customer wants to cancel because the "
                "monthly charges are too high."
            ),
            "similarity_score": 0.94,
        },
        {
            "customer_id": "1005",
            "note": (
                "Customer requested service termination "
                "after repeated connectivity problems."
            ),
            "similarity_score": 0.91,
        },
        {
            "customer_id": "1008",
            "note": (
                "Customer is unhappy with service quality "
                "and is considering cancellation."
            ),
            "similarity_score": 0.88,
        },
    ]

    return {
        "tool": "vector_search_tool",
        "status": "success",
        "query": query,
        "results": all_results[:num_results],
    }

##### Empty Vector Search Tool

In [0]:
def mock_empty_vector_search_tool(
    query: str,
    num_results: int,
) -> Dict[str, Any]:
    """
    Return a successful response with no matching notes.
    """

    return {
        "tool": "vector_search_tool",
        "status": "success",
        "query": query,
        "results": [],
    }

##### Failed Vector Search Tool

In [0]:
def mock_failed_vector_search_tool(
    query: str,
    num_results: int,
) -> Dict[str, Any]:
    """
    Simulate a Vector Search Tool failure.
    """

    return {
        "tool": "vector_search_tool",
        "status": "error",
        "query": query,
        "message": (
            "The Vector Search endpoint was unavailable."
        ),
    }

##### Invalid Vector Search Tool

In [0]:
def mock_invalid_vector_search_tool(
    query: str,
    num_results: int,
) -> Dict[str, Any]:
    """
    Simulate a malformed Vector Search response.
    """

    return {
        "tool": "vector_search_tool",
        "status": "success",
        "query": query,
        # "results" intentionally omitted.
    }

##### Invalid Search Result Tool

In [0]:
def mock_invalid_search_result_tool(
    query: str,
    num_results: int,
) -> Dict[str, Any]:
    """
    Simulate a response containing an invalid individual result.
    """

    return {
        "tool": "vector_search_tool",
        "status": "success",
        "query": query,
        "results": [
            {
                "customer_id": "1001",
                # "note" intentionally omitted.
                "similarity_score": 0.94,
            }
        ],
    }

##### 20. Helper for Mock Coordinator Results

In [0]:
def create_mock_vector_search_coordinator_result(
    task_description: str,
) -> CoordinatorResult:
    """
    Create a Coordinator result containing one Vector Search task.
    """

    return CoordinatorResult(
        status=SUCCESS_STATUS,
        message="Execution plan created successfully.",
        request_type="vector_search",
        reasoning=(
            "The user is asking for information that should "
            "be retrieved from customer notes using semantic search."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name=VECTOR_SEARCH_AGENT_NAME,
                task_description=task_description,
                depends_on=[],
            )
        ],
    )

##### 21. Test the Vector Search Agent

In [0]:
def test_vector_search_agent() -> None:
    """
    Run unit tests for the Vector Search Agent.
    """

    print("=" * 80)
    print("TEST 1: Successful semantic search")
    print("=" * 80)

    state = create_initial_state(
        "Why are customers likely to cancel service?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find customer notes explaining why customers "
                "are likely to cancel service."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_successful_vector_search_tool
        ),
        num_results=3,
    )

    assert VECTOR_SEARCH_AGENT_NAME in (
        updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][VECTOR_SEARCH_AGENT_NAME]

    assert result.status == SUCCESS_STATUS
    assert len(result.results) == 3
    assert len(updated_state["errors"]) == 0

    print("PASS")
    print(result.model_dump())
    print()


    print("=" * 80)
    print("TEST 2: Successful search with no results")
    print("=" * 80)

    state = create_initial_state(
        "Find notes about an unknown service issue."
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find customer notes about an unknown "
                "service issue."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_empty_vector_search_tool
        ),
    )

    result = updated_state[
        "agent_results"
    ][VECTOR_SEARCH_AGENT_NAME]

    assert result.status == SUCCESS_STATUS
    assert len(result.results) == 0
    assert len(updated_state["errors"]) == 0
    assert "no similar" in result.message.lower()

    print("PASS")
    print(result.model_dump())
    print()


    print("=" * 80)
    print("TEST 3: Vector Search Agent skips")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = CoordinatorResult(
        status=SUCCESS_STATUS,
        message="Execution plan created successfully.",
        request_type="sql_analytics",
        reasoning=(
            "The user is requesting a structured SQL calculation."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name="sql_agent",
                task_description=(
                    "Count the number of churned customers."
                ),
                depends_on=[],
            )
        ],
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_successful_vector_search_tool
        ),
    )

    assert VECTOR_SEARCH_AGENT_NAME not in (
        updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 0

    assert (
        updated_state["execution_history"][-1]["status"]
        == SKIPPED_STATUS
    )

    print("PASS")
    print(
        updated_state["execution_history"][-1]
    )
    print()


    print("=" * 80)
    print("TEST 4: Vector Search Tool failure")
    print("=" * 80)

    state = create_initial_state(
        "Why are customers unhappy?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find customer notes explaining customer "
                "dissatisfaction."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_failed_vector_search_tool
        ),
    )

    assert VECTOR_SEARCH_AGENT_NAME not in (
        updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    assert (
        updated_state["execution_history"][-1]["status"]
        == ERROR_STATUS
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()


    print("=" * 80)
    print("TEST 5: Missing results field")
    print("=" * 80)

    state = create_initial_state(
        "Why do customers cancel?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find notes explaining customer cancellations."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_invalid_vector_search_tool
        ),
    )

    assert VECTOR_SEARCH_AGENT_NAME not in (
        updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    assert "results" in (
        updated_state["errors"][-1][
            "error_message"
        ].lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()


    print("=" * 80)
    print("TEST 6: Invalid individual search result")
    print("=" * 80)

    state = create_initial_state(
        "Why do customers terminate service?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find notes explaining why customers "
                "terminate service."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_invalid_search_result_tool
        ),
    )

    assert VECTOR_SEARCH_AGENT_NAME not in (
        updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    assert "note" in (
        updated_state["errors"][-1][
            "error_message"
        ].lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()


    print("=" * 80)
    print("ALL VECTOR SEARCH AGENT TESTS PASSED")
    print("=" * 80)

##### 22. Run the Tests

In [0]:
test_vector_search_agent()

##### 23. Schema Verification

In [0]:
print(
    "CoordinatorResult:",
    CoordinatorResult.model_fields.keys(),
)

print(
    "AgentTask:",
    AgentTask.model_fields.keys(),
)

print(
    "VectorSearchAgentResult:",
    VectorSearchAgentResult.model_fields.keys(),
)


##### 24. Key Learnings

##### The Coordinator decides when the agent runs

- The Vector Search Agent checks: state["coordinator_result"].execution_plan
- It runs only when a task is assigned to: "vector_search_agent"

##### Semantic search uses unstructured text

- The SQL Agent works with structured aggregate data.
- The Vector Search Agent works with customer-note text and retrieves semantically similar records.

##### Dependency injection separates the agent from the implementation

- The agent receives: vector_search_tool as a function argument.
- During testing, this is a mock function. During orchestration, it will be the real Databricks Vector Search function.

##### Empty results are not necessarily errors

- A successful tool response may contain: "results": []
- This means the search worked but found no relevant matches.
- A missing "results" field is a malformed response and should be treated as an error.

##### Tool results are validated before entering shared state

The agent validates:

- Tool status
- Result collection type
- Customer IDs
- Customer notes
- Similarity scores

Only validated results are stored in shared state.

25. Conclusion

The Vector Search Agent now:

- Reads the Coordinator execution plan.
- Finds the assigned Vector Search task.
- Validates task dependencies.
- Resolves the semantic-search query.
- Calls an injected Vector Search Tool.
- Validates and normalizes the tool response.
- Creates a validated VectorSearchAgentResult.
- Stores the result in shared state.
- Handles empty results.
- Handles malformed responses and tool failures.
- Skips cleanly when no Vector Search task is assigned.
- Can be unit tested independently with mock tools.

26. Next Notebook

The next notebook is: 07_retention_agent

The Retention Agent will:

- Read the Coordinator execution plan.
- Determine whether a retention recommendation is required.
- Validate its task dependencies.
- Read Prediction Agent results when required.
- Read Vector Search Agent results when required.
- Call the injected Retention Tool.
- Validate the Retention Tool response.
- Create a validated RetentionAgentResult.
- Store the result in shared state.
- Skip cleanly when no retention task is assigned.
- Record execution history and errors.